# Module 11 Guided Lab: Final Data Checks, Data Dictionary, and ABT Documentation

**BAN 6003: Data Management and Analytics Integration**

This module is the final preparation step before we apply analytics to an ABT.

We are not learning many new Pandas or SQL functions this week. Instead, we are using the skills we already learned to ask:

> Is this ABT ready for modeling, handoff, and business interpretation?

A good ABT should be checked, documented, and understandable to someone other than the person who built it.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Conduct final sanity checks on a prepared ABT.
2. Check missing values, impossible values, duplicate entity rows, and target variable issues.
3. Create a simple data dictionary for an ABT.
4. Write a transformation and integration summary.
5. Prepare a data handoff package.
6. Apply this same process to your semester project ABT.

Starting next module, we will apply analytics and modeling. That means this is the moment to slow down and make sure the ABT is trustworthy.

## Business Scenario

Imagine you are handing your ABT to another analyst or to your future self.

That person needs to understand:

- what one row means
- where the columns came from
- what transformations were applied
- what the target variable means
- what missing values or limitations still exist
- whether the table is safe to use for modeling

If they cannot understand these points, the ABT is not ready for handoff.

## 0. Setup

This lab uses a small demonstration ABT that the notebook builds from `nycflights13`. You do not need to load any other data.

For the demo, we will build a route-level ABT with one row per origin-destination route.

After the lab, apply the same documentation steps to your own project ABT in your project repository. That work is separate from this notebook.

### Package setup note

You can work in GitHub Codespaces or in local VS Code. On the UTSA campus network Codespaces may not connect; use the local setup there (see the Canvas page on local setup).

If you are using **GitHub Codespaces**, the required packages should already be installed from `requirements.txt` when the Codespace is created. You usually do not need to run any `%pip install` command.

If you are using **local VS Code** and an import fails, uncomment and run the `%pip install` line(s) in the setup cell below, then rerun the imports.


In [1]:
# Codespaces: nycflights13 is installed from requirements.txt.
# Local only: if the import below fails, uncomment and run this line:
# %pip install nycflights13 "setuptools<81"
# (setuptools<81 is needed because nycflights13 uses pkg_resources, which newer setuptools removed.)

import nycflights13
import pandas as pd
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

output_dir = repo_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)


/opt/conda/lib/python3.12/site-packages/nycflights13/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename as _rf


In [2]:
flights = nycflights13.flights.copy()
airports = nycflights13.airports.copy()
airlines = nycflights13.airlines.copy()

## 1. Build a Demo ABT

For this demonstration, we will create a route-level ABT.

**Unit of analysis:** one row per origin-destination route.

This ABT is not meant to be perfect. It is a realistic practice table for final checks and documentation.

In [3]:
route_abt = (
    flights
    .groupby(["origin", "dest"])
    .agg(
        flight_count=("flight", "count"),
        avg_dep_delay=("dep_delay", "mean"),
        avg_arr_delay=("arr_delay", "mean"),
        max_arr_delay=("arr_delay", "max"),
        avg_distance=("distance", "mean"),
        avg_air_time=("air_time", "mean")
    )
    .reset_index()
)

route_abt.head()

,origin,dest,flight_count,avg_dep_delay,avg_arr_delay,max_arr_delay,avg_distance,avg_air_time
0,EWR,ALB,439,23.620525,14.397129,328.0,143.0,31.787081
1,EWR,ANC,8,12.875000,-2.500000,39.0,3370.0,413.125000
2,EWR,ATL,5022,15.501738,13.233183,796.0,746.0,111.993847
3,EWR,AUS,968,11.508863,-0.474399,349.0,1504.0,211.247649
4,EWR,AVL,265,8.616601,8.804781,228.0,583.0,89.796813


Add destination airport name to make the table more understandable.

This is a left merge because the route-level ABT is our main table and we want to preserve its rows.

In [4]:
dest_airports = airports[["faa", "name", "lat", "lon"]].rename(
    columns={
        "faa": "dest",
        "name": "dest_airport_name",
        "lat": "dest_lat",
        "lon": "dest_lon"
    }
)

route_abt = route_abt.merge(dest_airports, how="left", on="dest")

route_abt.head()

,origin,dest,flight_count,avg_dep_delay,avg_arr_delay,max_arr_delay,avg_distance,avg_air_time,dest_airport_name,dest_lat,dest_lon
0,EWR,ALB,439,23.620525,14.397129,328.0,143.0,31.787081,Albany Intl,42.748267,-73.801692
1,EWR,ANC,8,12.875000,-2.500000,39.0,3370.0,413.125000,Ted Stevens Anchorage Intl,61.174361,-149.996361
2,EWR,ATL,5022,15.501738,13.233183,796.0,746.0,111.993847,Hartsfield Jackson Atlanta Intl,33.636719,-84.428067
3,EWR,AUS,968,11.508863,-0.474399,349.0,1504.0,211.247649,Austin Bergstrom Intl,30.194528,-97.669889
4,EWR,AVL,265,8.616601,8.804781,228.0,583.0,89.796813,Asheville Regional Airport,35.436194,-82.541806


## 2. Final Sanity Check 1: Row Meaning and Shape

Before checking values, first confirm what one row means.

For this ABT:

> One row should represent one origin-destination route.

The key columns should be `origin` and `dest`.

In [5]:
route_abt.shape

(224, 11)

In [6]:
route_abt[["origin", "dest"]].head()

,origin,dest
0,EWR,ALB
1,EWR,ANC
2,EWR,ATL
3,EWR,AUS
4,EWR,AVL


### Check duplicate entity rows

If one row should represent one route, then each origin-destination pair should appear only once.

In [7]:
route_abt[["origin", "dest"]].duplicated().sum()

np.int64(0)

If this number is greater than zero, the ABT has duplicate entity rows and should not be used for modeling until the issue is resolved.

### Your Turn 1

Write one sentence explaining what one row means in this ABT.

Then identify the key column or columns that define one row.

**Your answer:**  
One row in the ABT represents one origin-destination flight route, with summarized information about the flights on that route. The key columns are `origin` and `dest`, which together uniquely identify each route.

## 3. Final Sanity Check 2: Missing Values

Missing values are not always wrong, but they must be understood.

For an ABT, missing values can affect:

- modeling
- interpretation
- business recommendations
- whether a column should be used at all

In [8]:
missing_summary = route_abt.isna().sum().reset_index()
missing_summary.columns = ["column_name", "missing_count"]
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(route_abt) * 100).round(1)

missing_summary.sort_values("missing_count", ascending=False)

,column_name,missing_count,missing_percent
8,dest_airport_name,7,3.1
10,dest_lon,7,3.1
9,dest_lat,7,3.1
4,avg_arr_delay,1,0.4
5,max_arr_delay,1,0.4
7,avg_air_time,1,0.4
3,avg_dep_delay,1,0.4
2,flight_count,0,0.0
1,dest,0,0.0
0,origin,0,0.0


### Questions to ask

For any column with missing values:

- Is this missingness expected?
- Did it come from the raw data?
- Did it come from an integration step?
- Should the column be kept, dropped, or imputed later?
- Should the limitation be documented?

This module is about checking and documenting, not making every dataset perfect.

### Your Turn 2

Identify one column with missing values, or confirm that no important columns have missing values.

Write one short note explaining what you would do next.

**Your answer:**  
The destination airport name, latitude, and longitude each have 7 missing values (3.1%). I would investigate whether these missing values came from the airport data or the merge. I would document the missing values before deciding whether to keep or remove the affected rows.

## 4. Final Sanity Check 3: Impossible or Suspicious Values

Some values may be technically non-missing but still suspicious.

Examples:

- negative flight counts
- negative distances
- impossible average air times
- unusually large delay values

Not every extreme value is wrong. But it should be noticed.

In [9]:
route_abt.describe()

,flight_count,avg_dep_delay,avg_arr_delay,max_arr_delay,avg_distance,avg_air_time,dest_lat,dest_lon
count,224.000000,223.000000,223.000000,223.000000,224.000000,223.000000,217.000000,217.000000
mean,1503.464286,14.460038,7.803616,367.802691,987.339773,143.228092,37.151552,-89.151989
std,1851.848985,7.076796,8.766000,220.813783,778.512987,97.771737,5.787763,15.535549
min,1.000000,-9.000000,-22.000000,-22.000000,17.000000,25.466019,21.318681,-157.922428
25%,289.250000,10.102531,2.874643,250.500000,440.750000,73.305461,33.636719,-94.713905
50%,765.500000,13.664606,7.517555,334.000000,754.000000,115.372401,38.174086,-84.219375
75%,2061.000000,18.783245,12.612513,413.500000,1376.250000,195.040787,41.533972,-80.040528
max,11262.000000,41.818471,44.585106,1272.000000,4983.000000,623.087719,61.174361,-68.828139


In [10]:
# Basic impossible-value checks
checks = {
    "negative_flight_count": (route_abt["flight_count"] < 0).sum(),
    "zero_or_negative_distance": (route_abt["avg_distance"] <= 0).sum(),
    "zero_or_negative_air_time": (route_abt["avg_air_time"] <= 0).sum(),
}

checks

{'negative_flight_count': np.int64(0),
 'zero_or_negative_distance': np.int64(0),
 'zero_or_negative_air_time': np.int64(0)}

### Check extreme values

Now sort to inspect routes with very high average arrival delay.

In [11]:
route_abt.sort_values("avg_arr_delay", ascending=False).head(10)

,origin,dest,flight_count,avg_dep_delay,avg_arr_delay,max_arr_delay,avg_distance,avg_air_time,dest_airport_name,dest_lat,dest_lon
13,EWR,CAE,104,36.336842,44.585106,224.0,602.0,92.148936,Columbia Metropolitan,33.938833,-81.119528
84,EWR,TYS,323,41.818471,41.150160,281.0,631.0,97.635783,Mc Ghee Tyson,35.810972,-83.994028
82,EWR,TUL,315,34.906355,33.659864,262.0,1215.0,178.309524,Tulsa Intl,36.198389,-95.888111
53,EWR,OKC,346,30.568807,30.619048,262.0,1325.0,193.009524,Will Rogers World,35.393089,-97.600733
36,EWR,JAC,23,28.700000,29.894737,175.0,1874.0,263.263158,Jackson Hole Airport,43.607333,-110.737750
65,EWR,RIC,1667,26.478960,25.820433,463.0,277.0,53.543653,Richmond Intl,37.505167,-77.319667
47,EWR,MKE,1094,23.551174,24.003781,441.0,725.0,115.372401,General Mitchell Intl,42.947222,-87.896583
23,EWR,DSM,411,29.328244,23.548718,322.0,1017.0,149.541026,Des Moines Intl,41.533972,-93.663083
48,EWR,MSN,354,21.437143,23.197101,337.0,799.0,124.246377,Dane Co Rgnl Truax Fld,43.139858,-89.337514
63,EWR,PWM,770,25.714674,23.005479,310.0,284.0,48.727397,Portland Intl Jetport,43.646161,-70.309281


Extreme values do not automatically mean errors. They may be real business signals. But they should be reviewed before modeling.

### Your Turn 3

Choose one numeric column and inspect its minimum, maximum, and average.

Write one sentence explaining whether the values look reasonable.

In [12]:
route_abt["avg_arr_delay"].agg(["min", "max", "mean"])

min    -22.000000
max     44.585106
mean     7.803616
Name: avg_arr_delay, dtype: float64

**Your answer:**  
The average arrival delay ranges from approximately -22.00 to 44.59 minutes, with an overall average of 7.80 minutes. These values appear reasonable because negative delays indicate flights arriving early, while positive delays indicate flights arriving late.

## 5. Final Sanity Check 4: Target Variable Issues

If the ABT will be used for modeling, the target variable must be clearly defined.

For this demo ABT, suppose our target is:

> average arrival delay by route

That means `avg_arr_delay` is the outcome we may want to explain or predict.

In [13]:
route_abt["avg_arr_delay"].isna().sum()

np.int64(1)

In [14]:
route_abt["avg_arr_delay"].describe()

count    223.000000
mean       7.803616
std        8.766000
min      -22.000000
25%        2.874643
50%        7.517555
75%       12.612513
max       44.585106
Name: avg_arr_delay, dtype: float64

### Questions to ask about the target

- What exactly does the target measure?
- Is the target missing for some rows?
- Is the target available at the time of prediction?
- Are there extreme values that should be reviewed?
- Would this target support the business question?

These questions help prevent modeling mistakes later.

### Your Turn 4

For your project ABT, identify your likely target variable.

If you do not know yet, identify one possible target variable.

Write 2–3 sentences explaining what it measures and whether it has any concerns.

**Your answer:**  
A possible target variable for my "Movies" project is the average user rating for each movie. This measures how highly viewers rated each movie and could help identify genres associated with higher ratings. One concern is that ratings only reflect users who chose to rate the movies, which may not represent the entire audience.

## 6. Create a Data Dictionary

A data dictionary explains what each column means.

At minimum, include:

- column name
- data type
- description
- source
- transformation or derivation
- notes or concerns

A data dictionary is not busywork. It is what makes your dataset usable by someone else.

In [15]:
data_dictionary = pd.DataFrame({
    "column_name": route_abt.columns,
    "data_type": [str(route_abt[col].dtype) for col in route_abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": ""
})

data_dictionary

,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,origin,str,,,,
1,dest,str,,,,
2,flight_count,int64,,,,
3,avg_dep_delay,float64,,,,
4,avg_arr_delay,float64,,,,
5,max_arr_delay,float64,,,,
6,avg_distance,float64,,,,
7,avg_air_time,float64,,,,
8,dest_airport_name,str,,,,
9,dest_lat,float64,,,,


Now fill in several rows manually as examples.

In a real project, you should complete this for every final ABT column.

In [16]:
example_cols = ["description", "source", "transformation_or_derivation", "notes_or_concerns"]

data_dictionary.loc[data_dictionary["column_name"] == "origin", example_cols] = [
    "Origin airport code for the route", "flights table", "Used as part of route-level grouping key",
    "None identified"
]

data_dictionary.loc[data_dictionary["column_name"] == "dest", example_cols] = [
    "Destination airport code for the route", "flights table", "Used as part of route-level grouping key",
    "Includes one unusual route, EWR to LGA, with a single cancelled flight"
]

data_dictionary.loc[data_dictionary["column_name"] == "flight_count", example_cols] = [
    "Number of flights observed for the route", "flights table", "Count of flight records grouped by origin and destination",
    "Ranges from 1 to 11,262; 10 routes have fewer than 10 flights, so their averages are unstable"
]

data_dictionary.loc[data_dictionary["column_name"] == "avg_arr_delay", example_cols] = [
    "Average arrival delay for flights on the route, in minutes", "flights table", "Mean of arr_delay grouped by origin and destination",
    "Missing for EWR to LGA (its only flight was cancelled); negative values mean early arrival on average"
]

data_dictionary.loc[data_dictionary["column_name"] == "dest_airport_name", example_cols] = [
    "Full name of the destination airport", "airports table", "Joined from airports using dest = faa",
    "Missing for 7 routes whose destination (BQN, PSE, SJU, STT) is not in the airports table"
]

data_dictionary


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,origin,str,Origin airport code for the route,flights table,Used as part of route-level grouping key,None identified
1,dest,str,Destination airport code for the route,flights table,Used as part of route-level grouping key,"Includes one unusual route, EWR to LGA, with a..."
2,flight_count,int64,Number of flights observed for the route,flights table,Count of flight records grouped by origin and ...,"Ranges from 1 to 11,262; 10 routes have fewer ..."
3,avg_dep_delay,float64,,,,
4,avg_arr_delay,float64,Average arrival delay for flights on the route...,flights table,Mean of arr_delay grouped by origin and destin...,Missing for EWR to LGA (its only flight was ca...
5,max_arr_delay,float64,,,,
6,avg_distance,float64,,,,
7,avg_air_time,float64,,,,
8,dest_airport_name,str,Full name of the destination airport,airports table,Joined from airports using dest = faa,"Missing for 7 routes whose destination (BQN, P..."
9,dest_lat,float64,,,,


### Your Turn 5

The cell above documented 5 columns (`origin`, `dest`, `flight_count`, `avg_arr_delay`, `dest_airport_name`). Complete the data dictionary for the 6 remaining ABT columns. Each row should have a useful `description`, `source`, and `transformation_or_derivation`. In `notes_or_concerns`, enter a real concern or `None identified` rather than leaving the field blank.

Use the completed dictionary in the handoff-readiness audit later in the notebook.


In [17]:

example_cols = ["description", "source", "transformation_or_derivation", "notes_or_concerns"]

data_dictionary.loc[data_dictionary["column_name"] == "max_arr_delay", example_cols] = [
    "Maximum arrival delay for flights on the route, in minutes",
    "flights table",
    "Maximum arr_delay grouped by origin and destination",
    "Extreme delays should be reviewed"
]

data_dictionary.loc[data_dictionary["column_name"] == "avg_distance", example_cols] = [
    "Average flight distance for the route, in miles",
    "flights table",
    "Mean distance grouped by origin and destination",
    "None identified"
]

data_dictionary.loc[data_dictionary["column_name"] == "avg_air_time", example_cols] = [
    "Average time spent in the air, in minutes",
    "flights table",
    "Mean air_time grouped by origin and destination",
    "Missing for one route"
]

data_dictionary.loc[data_dictionary["column_name"] == "dest_lat", example_cols] = [
    "Latitude of the destination airport",
    "airports table",
    "Joined from airports using dest = faa",
    "Missing for 7 routes"
]

data_dictionary.loc[data_dictionary["column_name"] == "dest_lon", example_cols] = [
    "Longitude of the destination airport",
    "airports table",
    "Joined from airports using dest = faa",
    "Missing for 7 routes"
]

data_dictionary.loc[data_dictionary["column_name"] == "avg_dep_delay", example_cols] = [
    "Average departure delay for flights on the route, in minutes",
    "flights table",
    "Mean dep_delay grouped by origin and destination",
    "Missing for one route"
]

data_dictionary


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,origin,str,Origin airport code for the route,flights table,Used as part of route-level grouping key,None identified
1,dest,str,Destination airport code for the route,flights table,Used as part of route-level grouping key,"Includes one unusual route, EWR to LGA, with a..."
2,flight_count,int64,Number of flights observed for the route,flights table,Count of flight records grouped by origin and ...,"Ranges from 1 to 11,262; 10 routes have fewer ..."
3,avg_dep_delay,float64,Average departure delay for flights on the rou...,flights table,Mean dep_delay grouped by origin and destination,Missing for one route
4,avg_arr_delay,float64,Average arrival delay for flights on the route...,flights table,Mean of arr_delay grouped by origin and destin...,Missing for EWR to LGA (its only flight was ca...
5,max_arr_delay,float64,Maximum arrival delay for flights on the route...,flights table,Maximum arr_delay grouped by origin and destin...,Extreme delays should be reviewed
6,avg_distance,float64,"Average flight distance for the route, in miles",flights table,Mean distance grouped by origin and destination,None identified
7,avg_air_time,float64,"Average time spent in the air, in minutes",flights table,Mean air_time grouped by origin and destination,Missing for one route
8,dest_airport_name,str,Full name of the destination airport,airports table,Joined from airports using dest = faa,"Missing for 7 routes whose destination (BQN, P..."
9,dest_lat,float64,Latitude of the destination airport,airports table,Joined from airports using dest = faa,Missing for 7 routes


## 7. Transformation and Integration Summary

The data dictionary documents columns. The transformation summary documents the process.

A good transformation summary answers:

- What raw data was used?
- What cleaning or filtering was applied?
- What transformations were created?
- What aggregations changed the unit of analysis?
- What tables were joined?
- What checks were performed?
- What assumptions or limitations remain?

### Example Transformation Summary

For this demo ABT:

1. Started with the `flights` table from `nycflights13`.
2. Aggregated flight-level records to one row per origin-destination route.
3. Calculated flight count, average departure delay, average arrival delay, maximum arrival delay, average distance, and average air time.
4. Joined destination airport name and location from the `airports` table using `dest = faa`.
5. Checked duplicate route keys using `origin` and `dest`.
6. Checked missing values across ABT columns.
7. Reviewed basic impossible values and extreme delay values.
8. The target variable for later modeling may be `avg_arr_delay`, but this choice should be justified by the business question.

### Your Turn 6

Write a transformation and integration summary for your own project ABT.

If your project ABT is not ready yet, write a draft summary based on your planned workflow.

**Your transformation summary:**  
For my movies project, I used movie metadata, ratings, credits, and keywords data to create a preliminary analytical base table (ABT). I cleaned and transformed the relevant columns, including checking for missing values and duplicate records. I aggregated individual ratings by movie to calculate movie-level rating information and joined the relevant tables using `movieId` and `tmdbId`. This allowed me to combine individual ratings with movie-level information while maintaining one row per movie. I also checked the resulting ABT for duplicate movie keys, missing values, and other data quality issues. One limitation is that audience ratings only represent users who chose to rate movies, which may introduce bias. The ABT can help analyze differences in audience ratings across movie genres and support decisions about which genres to prioritize for marketing.

## 8. Prepare a Handoff Package

A basic ABT handoff package should include:

1. Final ABT file
2. Complete data dictionary
3. Transformation and integration summary
4. Validation checklist
5. README or short handoff note

For this lab, the generated files belong in the repository-level `outputs` folder. The first export may still contain blanks until you complete Your Turn 5. After you complete the offline audit in Section 10, the notebook re-exports the data dictionary and the validation checklist so the final files match your completed work.


In [18]:
route_abt_path = output_dir / "route_abt_module11.csv"
dictionary_path = output_dir / "route_abt_data_dictionary_module11.csv"

route_abt.to_csv(route_abt_path, index=False)
data_dictionary.to_csv(dictionary_path, index=False)

print("ABT saved to:", route_abt_path)
print("Data dictionary saved to:", dictionary_path)


ABT saved to: /workspaces/ban6003-week-11-shanzayniazi/outputs/route_abt_module11.csv
Data dictionary saved to: /workspaces/ban6003-week-11-shanzayniazi/outputs/route_abt_data_dictionary_module11.csv


### Create a simple validation checklist

This checklist records whether the most important checks were completed. The status of each check is calculated from the ABT and the data dictionary, so it is not typed by hand.

- `Complete` means the check was done and no problem was found, or the finding is a known item that only needs documentation (for example, missing values).
- `Needs attention` means the check found a problem that must be fixed or explained before handoff.
- `In progress` means the data dictionary still has blank documentation cells.

We save a first version now. After the offline audit in Section 10, the notebook rebuilds and re-exports the checklist, so the final file shows the final status.

In [19]:
def build_validation_checklist(abt, dictionary):
    """Build the validation checklist from the current ABT and data dictionary."""
    text_fields = ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
    dictionary_text = dictionary[text_fields].fillna("").astype(str)
    blank_dictionary_cells = int((dictionary_text.apply(lambda col: col.str.strip()) == "").sum().sum())
    dictionary_matches_abt = len(dictionary) == len(abt.columns)

    duplicate_keys = int(abt[["origin", "dest"]].duplicated().sum())
    missing_key_values = int(abt[["origin", "dest"]].isna().sum().sum())
    missing_cells = int(abt.isna().sum().sum())
    nonpositive_values = int(
        (abt["flight_count"] <= 0).sum()
        + (abt["avg_distance"] <= 0).sum()
        + (abt["avg_air_time"] <= 0).sum()
    )
    missing_target_values = int(abt["avg_arr_delay"].isna().sum())

    def status(passed):
        return "Complete" if passed else "Needs attention"

    if dictionary_matches_abt and blank_dictionary_cells == 0:
        dictionary_status = "Complete"
    elif dictionary_matches_abt:
        dictionary_status = "In progress"
    else:
        dictionary_status = "Needs attention"

    return pd.DataFrame({
        "check": [
            "Unit of analysis is clearly defined",
            "Key columns are identified",
            "Duplicate entity rows checked",
            "Missing values checked",
            "Impossible or suspicious values reviewed",
            "Target variable reviewed",
            "Data dictionary complete",
            "Transformation summary written",
        ],
        "status": [
            "Complete",
            status(missing_key_values == 0),
            status(duplicate_keys == 0),
            "Complete",
            status(nonpositive_values == 0),
            "Complete",
            dictionary_status,
            "Complete",
        ],
        "notes": [
            "One row per origin-destination route",
            f"origin + dest; {missing_key_values} missing key values",
            f"{duplicate_keys} duplicate origin-dest keys found with duplicated().sum()",
            f"{missing_cells} missing cells found with isna().sum(); document them in the data dictionary",
            f"{nonpositive_values} nonpositive flight counts, distances, or air times",
            f"avg_arr_delay reviewed as possible target; {missing_target_values} missing target values",
            f"{len(dictionary)} dictionary rows for {len(abt.columns)} ABT columns; {blank_dictionary_cells} blank documentation cells",
            "Demo summary written in Section 7; write your project version separately",
        ],
    })


validation_checklist = build_validation_checklist(route_abt, data_dictionary)
validation_checklist

,check,status,notes
0,Unit of analysis is clearly defined,Complete,One row per origin-destination route
1,Key columns are identified,Complete,origin + dest; 0 missing key values
2,Duplicate entity rows checked,Complete,0 duplicate origin-dest keys found with duplic...
3,Missing values checked,Complete,25 missing cells found with isna().sum(); docu...
4,Impossible or suspicious values reviewed,Complete,"0 nonpositive flight counts, distances, or air..."
5,Target variable reviewed,Complete,avg_arr_delay reviewed as possible target; 1 m...
6,Data dictionary complete,Complete,11 dictionary rows for 11 ABT columns; 0 blank...
7,Transformation summary written,Complete,Demo summary written in Section 7; write your ...


In [20]:
checklist_path = output_dir / "route_abt_validation_checklist_module11.csv"
validation_checklist.to_csv(checklist_path, index=False)
print("Validation checklist saved to:", checklist_path)


Validation checklist saved to: /workspaces/ban6003-week-11-shanzayniazi/outputs/route_abt_validation_checklist_module11.csv


## 9. Apply This to Your Project

This week, you should apply the same process to your semester project.

For your project, prepare:

1. A clear statement of your ABT unit of analysis.
2. A final validation checklist.
3. A data dictionary for your ABT.
4. A transformation and integration summary.
5. A clean handoff folder in GitHub.

This work prepares you for the next module, where we begin applying analytics to the ABT.

## 10. Offline Assignment - Handoff-Readiness Audit

An analyst on another team will use the route ABT next week and cannot ask you to explain undocumented details. Decide whether the handoff package is ready.

1. Finish the data dictionary so every ABT column has all required documentation fields.
2. Complete the audit summary below for duplicate keys, missing key values, nonpositive operational measures, missing target values, and undocumented cells.
3. Re-export the completed data dictionary and the updated validation checklist (the export cell below does both).
4. Save the audit as `outputs/route_abt_handoff_audit_module11.csv`.
5. Make a release decision based on the evidence. A real extreme delay can remain if it is documented; do not silently remove it.


In [21]:
# Offline Assignment - complete the TODO values.
# Replace each None below with code that computes the number, for example route_abt[...].some_check().sum().
# Hints:
# - Use .duplicated().sum() on ["origin", "dest"].
# - Select the two key columns before counting missing values.
# - Use Boolean conditions such as (route_abt["flight_count"] <= 0).sum().
# - For documentation, select the four text fields, strip whitespace, compare to "", then sum.
required_documentation_fields = [
    "description", "source", "transformation_or_derivation", "notes_or_concerns"
]

handoff_audit = pd.DataFrame([{
    "abt_rows": len(route_abt),
    "dictionary_rows": len(data_dictionary),
    "dictionary_matches_abt_columns": len(data_dictionary) == len(route_abt.columns),
    "duplicate_route_keys": route_abt[["origin", "dest"]].duplicated().sum(),
    "missing_key_values": route_abt[["origin", "dest"]].isna().sum().sum(),
    "nonpositive_flight_counts": (route_abt["flight_count"] <= 0).sum(),
    "nonpositive_average_distance": (route_abt["avg_distance"] <= 0).sum(),
    "nonpositive_average_air_time": (route_abt["avg_air_time"] <= 0).sum(),
    "missing_target_values": route_abt["avg_arr_delay"].isna().sum(),
    "undocumented_dictionary_cells": (
    data_dictionary[required_documentation_fields]
    .fillna("")
    .astype(str)
    .apply(lambda col: col.str.strip() == "")
    .sum()
    .sum()
),

}])

handoff_audit


,abt_rows,dictionary_rows,dictionary_matches_abt_columns,duplicate_route_keys,missing_key_values,nonpositive_flight_counts,nonpositive_average_distance,nonpositive_average_air_time,missing_target_values,undocumented_dictionary_cells
0,224,11,True,0,0,0,0,0,1,0


In [22]:
# Run this cell after every TODO above has been replaced with working code.
if handoff_audit.isna().any(axis=None):
    print("Complete every audit TODO before exporting the final handoff files.")
else:
    data_dictionary.to_csv(dictionary_path, index=False)

    # Rebuild the checklist so its status reflects the completed dictionary and audit.
    validation_checklist = build_validation_checklist(route_abt, data_dictionary)
    validation_checklist.to_csv(checklist_path, index=False)

    audit_path = output_dir / "route_abt_handoff_audit_module11.csv"
    handoff_audit.to_csv(audit_path, index=False)
    print("Completed data dictionary saved to:", dictionary_path)
    print("Updated validation checklist saved to:", checklist_path)
    print("Handoff audit saved to:", audit_path)
    display(validation_checklist)

Completed data dictionary saved to: /workspaces/ban6003-week-11-shanzayniazi/outputs/route_abt_data_dictionary_module11.csv
Updated validation checklist saved to: /workspaces/ban6003-week-11-shanzayniazi/outputs/route_abt_validation_checklist_module11.csv
Handoff audit saved to: /workspaces/ban6003-week-11-shanzayniazi/outputs/route_abt_handoff_audit_module11.csv


,check,status,notes
0,Unit of analysis is clearly defined,Complete,One row per origin-destination route
1,Key columns are identified,Complete,origin + dest; 0 missing key values
2,Duplicate entity rows checked,Complete,0 duplicate origin-dest keys found with duplic...
3,Missing values checked,Complete,25 missing cells found with isna().sum(); docu...
4,Impossible or suspicious values reviewed,Complete,"0 nonpositive flight counts, distances, or air..."
5,Target variable reviewed,Complete,avg_arr_delay reviewed as possible target; 1 m...
6,Data dictionary complete,Complete,11 dictionary rows for 11 ABT columns; 0 blank...
7,Transformation summary written,Complete,Demo summary written in Section 7; write your ...


### Handoff Decision

**Your response here:** Write 5-6 sentences that approve the handoff, approve it with conditions, or hold it for revision. Cite at least four audit results, identify one remaining limitation or target concern, and state what the receiving analyst should know before using the ABT for modeling.

I would approve the handoff with conditions because the ABT passed the main validation checks, but there are still some missing values to consider. The ABT contains 224 rows, and the data dictionary includes all 11 columns. The audit found no duplicate origin-destination keys, no missing key values, and no nonpositive flight counts, average distances, or average air times. However, the dataset contains 25 missing cells overall, including one missing value for the potential target variable, `avg_arr_delay`. Before using the ABT for modeling, the receiving analyst should review the missing values and decide how to handle the missing target value. The analyst should also review the documented data limitations and confirm that `avg_arr_delay` is appropriate for the business question.

## 11. Final Reflection

Write 5–7 sentences answering:

1. What does one row mean in your project ABT?
2. What are your key columns?
3. What is your likely target variable?
4. What was the most important validation check?
5. Which column needs the clearest explanation in the data dictionary?
6. What still needs to be improved before modeling?


**Your reflection:**  
In my movies project, each row in the analytical base table (ABT) represents one unique movie. The primary key is `movieId`, which uniquely identifies each movie in the dataset. My likely target variable is `high_user_rating`, which indicates whether an eligible movie has an average user rating of at least 3.75. The most important validation check was confirming that there were no duplicate movie IDs, ensuring that each movie appears only once in the ABT. The data dictionary provides information about each column, including its description, source, transformations, and potential data quality concerns. Before modeling, I would need to address the remaining missing values, particularly in the audience ratings and keywords data. I would also need to consider potential bias because audience ratings only represent users who chose to rate movies.

## 12. Save Your Work

Before submitting:

1. Complete all Your Turn sections and the handoff-readiness audit.
2. Complete the handoff decision and final reflection.
3. Restart the kernel and run all cells from top to bottom.
4. Confirm all four Module 11 files exist in `outputs`.
5. Save the notebook, then commit and push your work.

```bash
git add .
git commit -m "Complete Module 11 ABT documentation lab"
git push
```


## Optional Practice: Diagnose a Mini Handoff Package (Not Graded)

This short practice does **not** count toward your lab grade. Use it if you want another repetition of the Week 11 handoff-readiness workflow before applying the same checks to your project.

A teammate sends a small route ABT and data dictionary. Do not repair the files immediately. First build an evidence-based audit that identifies whether the package is ready for another analyst.


In [23]:
practice_abt = pd.DataFrame({
    "origin": ["JFK", "JFK", "JFK", "LGA"],
    "dest": ["ORD", "ATL", "ATL", "BOS"],
    "flight_count": [820, 760, 760, 640],
    "avg_arr_delay": [8.4, 11.2, 11.2, pd.NA],
})

practice_dictionary = pd.DataFrame({
    "column_name": ["origin", "dest", "flight_count", "avg_arr_delay"],
    "description": ["Origin airport code", "Destination airport code", "Number of flights", ""],
    "source": ["flights.origin", "flights.dest", "flights", "flights.arr_delay"],
    "transformation_or_derivation": ["Grouped field", "Grouped field", "COUNT(*)", "AVG(arr_delay)"],
    "notes_or_concerns": ["Part of route key", "Part of route key", "Should be positive", "May be missing"],
})

display(practice_abt)
display(practice_dictionary)


,origin,dest,flight_count,avg_arr_delay
0,JFK,ORD,820,8.4
1,JFK,ATL,760,11.2
2,JFK,ATL,760,11.2
3,LGA,BOS,640,<NA>


,column_name,description,source,transformation_or_derivation,notes_or_concerns
0,origin,Origin airport code,flights.origin,Grouped field,Part of route key
1,dest,Destination airport code,flights.dest,Grouped field,Part of route key
2,flight_count,Number of flights,flights,COUNT(*),Should be positive
3,avg_arr_delay,,flights.arr_delay,AVG(arr_delay),May be missing


### Practice Task

Create a one-row DataFrame named `practice_handoff_audit` containing:

1. Number of ABT rows.
2. Number of duplicate `origin`-`dest` keys.
3. Number of missing key values.
4. Number of missing `avg_arr_delay` target values.
5. Whether the dictionary has one row for every ABT column.
6. Total blank cells across `description`, `source`, `transformation_or_derivation`, and `notes_or_concerns`.

Hints:

- Use `.duplicated(["origin", "dest"]).sum()` for the composite key.
- Select both key columns before applying `.isna().sum().sum()`.
- For documentation blanks, select the four text fields, apply `.fillna("")`, convert to string, strip whitespace, compare with `""`, and sum twice.
- Audit first. A handoff decision should describe the evidence before recommending repairs.


In [24]:
# Optional practice workspace
# Replace None with a one-row DataFrame containing the six requested checks.
practice_handoff_audit = None


In [25]:
# Run this self-check after you create practice_handoff_audit.
if practice_handoff_audit is None:
    print("Complete the optional practice first, then rerun this self-check.")
else:
    expected_practice_audit = pd.DataFrame([{
        "abt_rows": 4,
        "duplicate_route_keys": 1,
        "missing_key_values": 0,
        "missing_target_values": 1,
        "dictionary_matches_abt_columns": True,
        "undocumented_dictionary_cells": 1,
    }])
    print("Your audit:")
    display(practice_handoff_audit)
    print("Expected audit:")
    display(expected_practice_audit)


Complete the optional practice first, then rerun this self-check.


### Think About the Result

This package should be held for revision: it contains a duplicate route key, a missing target value, and an undocumented dictionary cell. Notice that the dictionary row count alone passes, but completeness still fails. A useful handoff decision should name each issue, explain why it matters to the receiving analyst, and request a specific correction or documented exception.
